In [1]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from tqdm import tqdm
import pandas as pd
import time
import numpy as np

##Import data
csvfile = 'marketing_campaign_cleaned_NODUMMIES'
df = pd.read_csv(csvfile)

#Synthcity
import sys
import warnings

warnings.filterwarnings("ignore")

import synthcity.logger as log
from synthcity.plugins import Plugins
from synthcity.plugins.core.dataloader import GenericDataLoader

log.add(sink=sys.stderr, level="INFO")

#Fit GAN using ALL training data
from synthcity.plugins import Plugins

syn_model_full = Plugins().get("ctgan")

##########################FUNCTIONS#################################

def dummify_columns(df):
    # Dummify marital and educational
    dummify_marital = pd.get_dummies(df['Marital_Status'],prefix='marital')
    df = pd.concat([df, dummify_marital],axis=1)

    dummify_edu = pd.get_dummies(df['Education'],prefix='education')
    df = pd.concat([df, dummify_edu], axis=1)

    # Drop transformed cols
    df.drop(columns=['Marital_Status', 'Education'], inplace=True)
    
    return df

def train_and_evaluate(df_train, df_test):
    # Split into features and target for train and test datasets
    X_train = df_train.drop('Response', axis=1)
    y_train = df_train['Response']
    X_test = df_test.drop('Response', axis=1)
    y_test = df_test['Response']
    
    # Identify overlapping columns in train and test: note that the train set is only small (7.5p), columns may be missing
    # if we add only a little bit of synth data, there is a chance that same column is missing in synth data
    common_columns = set(X_train.columns).intersection(X_test.columns)
    
    # Keep only common columns in train and test
    X_train = X_train[common_columns]
    X_test = X_test[common_columns]
    

    # Train a Random Forest model
    model = RandomForestClassifier(n_estimators = 500, max_depth = None, 
                                   max_features = 'auto', criterion = 'gini', min_samples_split = 2)
    model.fit(X_train, y_train)

    # Make predictions
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    # Calculate metrics
    report = classification_report(y_test, y_pred, output_dict=True)
    roc_auc = roc_auc_score(y_test, y_prob)

    # Store metrics
    results = {
        'accuracy': report['accuracy'],
        'precision': report['1']['precision'],
        'recall': report['1']['recall'],
        'f1': report['1']['f1-score'],
        'auc-roc': roc_auc
    }

    return results





<stdin>:1:10: fatal error: 'omp.h' file not found
#include <omp.h>
         ^~~~~~~
1 error generated.


[KeOps] Warning : omp.h header is not in the path, disabling OpenMP.
[KeOps] Warning : Cuda libraries were not detected on the system ; using cpu only mode
2023-08-04 19:05:35,822 - Created a temporary directory at /var/folders/8w/flfvck1j6m77x3j6jf0384bw0000gn/T/tmp0tllxizr
2023-08-04 19:05:35,823 - Writing /var/folders/8w/flfvck1j6m77x3j6jf0384bw0000gn/T/tmp0tllxizr/_remote_module_non_scriptable.py


In [2]:
# OTHER extrinisc FASTER had randomstate = 4

# Define synthetic set sizes


countlen = len(df)*0.7*0.85

syn_sizes = [1, 0.5*countlen, 1*countlen,  3*countlen,  5*countlen, 
            8*countlen, 12*countlen, 18*countlen, 32*countlen, 48*countlen, 64*countlen]

n_iterations = 100  # Number of bootstrapping iterations
syn_model = Plugins().get('ctgan')

# Placeholder for the results
results = []

# Bootstrap iteration loop
for i in range(n_iterations):
    # Resample entire dataset
    start_time = time.time()
    
    # Perform train/test split
    df_train_main, df_test = train_test_split(df, test_size=0.3, random_state=i*114)
    df_train_1, df_train_2 = train_test_split(df_train_main, test_size=0.15, random_state=i*1144)

    # Resample
    #df_test_resample = resample(df_test,replace=True)
    #df_train_1_resample = resample(df_train_1,replace=True)
    #df_train_2_resample = resample(df_train_2,replace=True)
    
    loader = GenericDataLoader(df_train_1, target_column='Response')
    syn_model.fit(loader)
    # Loop through the different synthetic set sizes
    for size in syn_sizes:
        # Generate synthetic set of the current size based on resampled training data
        #syn_set = syn_model.generate(count=size,random_state=i*124).dataframe()
        syn_set = syn_model.generate(count=size,random_state=i*114).dataframe()

        print(syn_set['Response'].mean() * 100)

        # Add synthetic data to resampled training data
        df_train_combined = pd.concat([df_train_2, syn_set], axis=0)

        # Dummify train and test datasets
        df_train_combined = dummify_columns(df_train_combined)
        df_test_dummified = dummify_columns(df_test.copy())  # .copy() to avoid SettingWithCopyWarning

        # Train and evaluate
        metrics = train_and_evaluate(df_train_combined, df_test_dummified)

        # Store the results with additional information
        metrics['syn_size'] = size
        metrics['iteration'] = i
        results.append(metrics)

    end_time = time.time()
    execution_time = end_time - start_time
    print(f"Time: {execution_time} seconds")
    print(f"Iteration: {i} " )
    
# Convert to DataFrame
results_exc_df_1 = pd.DataFrame(results)

 37%|██████████████▉                         | 749/2000 [05:05<08:29,  2.46it/s]


0.0
13.282442748091603
15.713196033562166
14.027954256670903
14.239975606037506
14.263935207241543
14.209489932033284
14.308702096125344
14.341519186337326
14.17727951217963
14.218850487101923
Time: 548.8929369449615 seconds
Iteration: 0 


 57%|██████████████████████▍                | 1149/2000 [08:20<06:10,  2.30it/s]


0.0
12.67175572519084
12.967200610221205
13.621346886912326
13.721603903034
13.882801333968557
13.428190306803023
13.38132542875291
13.70316556701522
13.764410709181568
13.557868661124742
Time: 732.2524328231812 seconds
Iteration: 1 


 27%|██████████▉                             | 549/2000 [04:09<10:58,  2.20it/s]


...

0.0
14.961832061068703
14.111365369946604
14.23125794155019
14.575392590333891
14.416388756550738
14.806580702534461
14.439974592420072
14.365338351237405
14.539333693270237
14.46656980206274
Time: 976.5844447612762 seconds
Iteration: 97 


 37%|██████████████▉                         | 749/2000 [06:50<11:26,  1.82it/s]


0.0
14.80916030534351
12.890922959572846
13.926302414231259
14.926055801189205
14.464030490709861
14.482627199390205
14.639000635189497
14.725007741228593
14.825165941499666
14.989400471619465
Time: 708.0109050273895 seconds
Iteration: 98 


 32%|████████████▉                           | 649/2000 [05:57<12:23,  1.82it/s]


0.0
14.961832061068703
14.492753623188406
14.155019059720459
13.67586522335722
14.787994282991901
14.444515022549705
14.143552826593266
14.384393683157468
14.571092831962398
14.459424052592714
Time: 696.4003040790558 seconds
Iteration: 99 


In [ ]:
results_exc_df_1.to_clipboard()